# Fast Tokenizers in the QA Pipeline  
## Simple Project Notebook (Easy English)

In this notebook, we will learn how a **Question Answering (QA)** system works using Hugging Face Transformers.

We will go step by step:

- Use the QA pipeline
- Understand how tokenization works
- See how the model finds answers
- Use offsets to extract text
- Get top-k answers
- Handle long contexts properly

---

## What you will learn

After this notebook, you will be able to:

- Use QA models easily
- Understand start and end logits
- Extract answers manually
- Work with long documents
- Build your own QA function

---

## 1. Setup

In [1]:
# Uncomment if needed
# !pip install -q transformers

In [2]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
from pprint import pprint

## 2. Load QA pipeline

In [3]:
qa_pipeline = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--distilbert--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is n

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


## 3. Simple example

In [4]:
context = '''
Transformers is backed by Jax, PyTorch, and TensorFlow.
'''

question = "Which libraries support Transformers?"

In [5]:
result = qa_pipeline(question=question, context=context)
pprint(result)

{'answer': 'Jax, PyTorch, and TensorFlow',
 'end': 55,
 'score': 0.7826315226993756,
 'start': 27}


The output shows:
- answer
- start position
- end position
- confidence score

## 4. Long context example

In [6]:
long_context = "Transformers supports many tasks. It is backed by Jax, PyTorch and TensorFlow."

In [7]:
pprint(qa_pipeline(question=question, context=long_context))

{'answer': 'Jax, PyTorch and TensorFlow',
 'end': 77,
 'score': 0.9477802096880623,
 'start': 50}


QA pipeline can handle long text automatically.

# Part 2: How it works inside

## 5. Load model and tokenizer

In [8]:
model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

## 6. Tokenization

In [9]:
inputs = tokenizer(question, context, return_tensors="pt")
inputs

{'input_ids': tensor([[  101,  5979,  9818,  1619, 25267,   136,   102, 25267,  1110,  5534,
          1118, 13612,   117,   153,  1183,  1942,  1766,  1732,   117,  1105,
          5157, 21484,  2271,  6737,   119,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [10]:
outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits

print(start_logits.shape)
print(end_logits.shape)

torch.Size([1, 26])
torch.Size([1, 26])


The model gives:
- start_logits → where answer starts
- end_logits → where answer ends

## 7. Mask unwanted tokens

In [11]:
sequence_ids = inputs.sequence_ids()

mask = [i != 1 for i in sequence_ids]
mask[0] = False

mask = torch.tensor(mask)[None]

start_logits[mask] = -10000
end_logits[mask] = -10000

## 8. Convert to probabilities

In [12]:
start_probs = torch.nn.functional.softmax(start_logits, dim=-1)[0]
end_probs = torch.nn.functional.softmax(end_logits, dim=-1)[0]

## 9. Find best answer span

In [13]:
scores = start_probs[:, None] * end_probs[None, :]
scores = torch.triu(scores)

idx = scores.argmax().item()

start_idx = idx // scores.shape[1]
end_idx = idx % scores.shape[1]

print(start_idx, end_idx)

11 23


## 10. Convert to real text using offsets

In [14]:
inputs_offsets = tokenizer(question, context, return_offsets_mapping=True)
offsets = inputs_offsets["offset_mapping"]

start_char, _ = offsets[start_idx]
_, end_char = offsets[end_idx]

answer = context[start_char:end_char]

print(answer)

Jax, PyTorch, and TensorFlow


# Part 3: Top-K answers

In [15]:
def get_top_k(start_probs, end_probs, offsets, context, k=3):
    scores = start_probs[:, None] * end_probs[None, :]
    scores = torch.triu(scores)

    flat_scores = scores.flatten()
    top_scores, top_idx = torch.topk(flat_scores, k)

    results = []
    for score, idx in zip(top_scores, top_idx):
        s = idx // scores.shape[1]
        e = idx % scores.shape[1]

        start_char, _ = offsets[s]
        _, end_char = offsets[e]

        results.append({
            "answer": context[start_char:end_char],
            "score": float(score)
        })

    return results

In [16]:
top_answers = get_top_k(start_probs, end_probs, offsets, context, k=3)
pprint(top_answers)

[{'answer': 'Jax, PyTorch, and TensorFlow', 'score': 0.782494306564331},
 {'answer': 'PyTorch, and TensorFlow', 'score': 0.1877889484167099},
 {'answer': 'Jax, PyTorch, and TensorFlow.', 'score': 0.015511888079345226}]


C:\Users\User\AppData\Local\Temp\ipykernel_20436\53573965.py:18: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  "score": float(score)


# Part 4: Long context handling

If text is too long, we split it into chunks.

In [17]:
inputs = tokenizer(
    question,
    long_context,
    max_length=384,
    truncation="only_second",
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="longest",
    return_tensors="pt"
)

In [18]:
offsets = inputs.pop("offset_mapping")
_ = inputs.pop("overflow_to_sample_mapping")

outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits

Each chunk gives its own answer.  
We select the best one.

# Final Summary

Key ideas:

- QA model predicts start and end positions
- We use softmax to get probabilities
- We calculate span scores
- Offsets help get real text
- Long text is split into chunks

---

Now you understand both:
- how to USE QA pipeline
- how it WORKS internally